In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd
import numpy as np
from copy import deepcopy
from collections import defaultdict
from tqdm import tqdm_notebook

In [ ]:
model_name = "Qwen/Qwen2-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')

In [ ]:
df = pd.read_csv("/content/fake_quotes - kabir.csv")

In [ ]:
df = df.rename(
    {
        "Fake_Quote": "quote",
        "Author": "ideology"
    },axis=1,
)

In [ ]:
def convert_to_chat_prompt(quote):
    messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot",
    },
    {"role": "user", "content": "Which famous person said the line '{}' Reply with the name else say that you do not know."},
    ]

    def _insert_quote(quote):
        temp = deepcopy(messages)
        temp[1]['content'] = temp[1]['content'].format(quote)
        return temp
    return [_insert_quote(x) for x in quote]


In [ ]:
records = defaultdict(list)
with torch.no_grad():
    model.eval()
    for chunk  in tqdm_notebook(np.array_split(df, 50)):
        quotes = chunk.quote.values
        ideology = chunk.ideology.values
        prompt = tokenizer.apply_chat_template(convert_to_chat_prompt(quotes),
                                            add_generation_prompt=True, return_tensors="pt",
                                            padding_side='left', padding=True,
                                            return_attention_mask=True).to(model.device)
        attention_mask = torch.ones(prompt.shape, dtype=torch.long, device=prompt.device)
        input_length = prompt.shape[1]
        generated_ids = model.generate(prompt, do_sample=False, max_new_tokens=10,
                                    attention_mask=attention_mask)
        response = tokenizer.batch_decode(generated_ids[:, input_length:], skip_special_tokens=True)
        for i, idea in enumerate(ideology):
            records[idea].append(response[i])

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
<ipython-input-34-d7a997a51a15>:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for chunk  in tqdm_notebook(np.array_split(df, 50)):


  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


In [ ]:
import pickle
with open("records_dump.pkl", "wb") as f:
    pickle.dump(records, f)